# This Is a Small Example Using Fracnetics

## Install and Load Packages

In [1]:
!pip install --upgrade fracnetics
!pip install pyvis
import fracnetics as fn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt
import copy
from sklearn.metrics import accuracy_score
from pyvis.network import Network # for network visualization 
from IPython.display import HTML # for network visualization

## Loading Data (Iris)

In [2]:
data = load_iris(as_frame=True)

In [3]:
X, y = data['data'], data['target']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=45,shuffle=True,stratify=y)
X_train=copy.deepcopy(X_train)
y_train=copy.deepcopy(y_train)

In [4]:
X_train.describe()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
count,135.000000,135.000000,135.000000,135.000000
mean,5.865926,3.064444,3.774074,1.195556
std,0.837922,0.435456,1.790396,0.758422
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.550000,0.300000
50%,5.800000,3.000000,4.400000,1.300000
75%,6.400000,3.350000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


## Initializing the Population

In [5]:
# initializing population

pop = fn.Population(
    seed=42,
    ni=300,
    jn=10,
    jnf=4,
    pn=10,
    pnf=3,
    fractalJudgment=False,
    nFeatureValues=[0,0,0,0]
)

In [6]:
minFeatures = X_train.min(axis=0).values
maxFeatures = X_train.max(axis=0).values

In [7]:
maxFeatures

array([7.9, 4.4, 6.9, 2.5])

In [8]:
minFeatures

array([4.3, 2. , 1. , 0.1])

In [9]:
minFeatures = [4.3,2,1,0.1] # TODO 
maxFeatures = [7.9,4.4,6.9,2.5]

In [10]:
pop.setAllNodeBoundaries(minFeatures,maxFeatures)

## Training the Population

In [11]:
fitnessProgess = []
for g in range(200):
    pop.callTraversePath(X_train.values.astype('float32'),dMax=10)
    for ind in pop.individuals:
        ind.fitness = accuracy_score(y_train,ind.decisions)
    pop.tournamentSelection(2,1)
    pop.callAddDelNodes(minFeatures,maxFeatures, 0.1)
    pop.callEdgeMutation(0.05, 0.05)
    pop.crossover(1, "randomWidth")
  
    print(f"Generation: {g} | Best Fitness: {pop.bestFit}")
    fitnessProgess.append([ind.fitness for ind in pop.individuals]) # append fitness of each individual for boxplot chart

Generation: 0 | Best Fitness: 0.4888888895511627
Generation: 1 | Best Fitness: 0.6518518328666687
Generation: 2 | Best Fitness: 0.6518518328666687
Generation: 3 | Best Fitness: 0.6592592597007751
Generation: 4 | Best Fitness: 0.6592592597007751
Generation: 5 | Best Fitness: 0.6592592597007751
Generation: 6 | Best Fitness: 0.6592592597007751
Generation: 7 | Best Fitness: 0.6592592597007751
Generation: 8 | Best Fitness: 0.6592592597007751
Generation: 9 | Best Fitness: 0.7185184955596924
Generation: 10 | Best Fitness: 0.7333333492279053
Generation: 11 | Best Fitness: 0.7333333492279053
Generation: 12 | Best Fitness: 0.7333333492279053
Generation: 13 | Best Fitness: 0.7333333492279053
Generation: 14 | Best Fitness: 0.7333333492279053
Generation: 15 | Best Fitness: 0.7333333492279053
Generation: 16 | Best Fitness: 0.7333333492279053
Generation: 17 | Best Fitness: 0.7333333492279053
Generation: 18 | Best Fitness: 0.7333333492279053
Generation: 19 | Best Fitness: 0.7333333492279053
Generation

KeyboardInterrupt: 

In [ ]:
# Plot fitness progression
plt.figure(figsize=(15, 6))
plt.boxplot(fitnessProgess, whis=2, sym=".")
plt.title("Fitness Progress")
plt.xlabel("Generation")
plt.ylabel("Fitness")
ticks = range(0,len(fitnessProgess), 10)
plt.xticks(ticks,labels = ticks)
plt.show()

In [ ]:
print(f"Train Fitness: {pop.bestFit}")
ind = pop.individuals[-1]
ind.traversePath(X_test.values,10)
acc = accuracy_score(y_test,ind.decisions)
print(f"Test Fitness: {acc}")

In [ ]:
pop.individuals[-1].fitness
print(f"Start Node: {pop.individuals[-1].startNode.edges}")
for node in pop.individuals[-1].innerNodes:
  marker = ""
  if node.used == False:
    marker = "*"
  print(f"{marker} ID: {node.id} Type: {node.type} | Function: {node.f} Edges: {node.edges} | Boundaries: {node.boundaries}")

In [ ]:
net = Network(notebook=True, directed=True, cdn_resources="in_line")
net.force_atlas_2based()
individual = pop.individuals[-1]
processingFunctionNames = ["Setosa", "Versicolour", "Virginica"]
judgmentFunctionNames = ["sepal length", "sepal width", "petal length", "petal width"]

# adding start node
net.add_node(-1, label=individual.startNode.type, color = "#635b3e")

# adding inner nodes
for node in individual.innerNodes:
    # set color
    if node.type == "J":
        color = "#3e6341"
        net.add_node(node.id, label=f"F: {judgmentFunctionNames[node.f]}", color=color)
    elif node.type == "P":
        color = "#372f61"
        net.add_node(node.id, label=f"F: {processingFunctionNames[node.f]}", color=color)
    else:
        color = None
        
for node in individual.innerNodes:
    for idx, edge in enumerate(node.edges):
        if node.type == "J":
            edgeLabel = f"{node.boundaries[idx]} bis {node.boundaries[idx+1]}"
            net.add_edge(node.id, edge, title = edgeLabel,
                    font={'size': 14, 'color': '#3e6341', 'align': 'horizontal'})
        else:
            net.add_edge(node.id, edge,
                    font={'size': 14, 'color': '#372f61', 'align': 'horizontal'})
        
# adding start node edge 
net.add_edge(-1, individual.startNode.edges[0])
net.save_graph("lunarlander_solution.html")
HTML(filename="lunarlander_solution.html")

## Store and Load Population

After training the population we can store and load them as a .pkl file via the two function:

- storePopulation() and 
- loadPopulation()

In [ ]:
fn.storePopulation(pop, "model1.pkl")

In [ ]:
loadedPopulation = fn.loadPopulation("model1.pkl")
loadedPopulation.individuals[-1].fitness